# Comparativa de Agencias de Transporte
Unifica las categorías de todas las agencias y encuentra la más económica por producto.

In [1]:
import openpyxl
import pandas as pd
import re
import warnings
warnings.filterwarnings('ignore')

wb_libro = openpyxl.load_workbook('Libro1.xlsx')
ws_libro = wb_libro['Hoja1']

wb_ag = openpyxl.load_workbook('COMPARATIVA AGENCIAS.xlsx', data_only=True)
print('Hojas disponibles:', wb_ag.sheetnames[:5], '...')

Hojas disponibles: ['Costo ag. Mayo 2026', 'Costo ag. Abril 2026', 'Costo ag. Marzo 2026', 'Costo ag. Febrero 2026', 'Costo.ag Enero 2026'] ...


In [2]:
# ── Cargar Libro1 ──────────────────────────────────────────────────────────
rows = list(ws_libro.iter_rows(values_only=True))
cols = rows[0]
df = pd.DataFrame(rows[1:], columns=cols)
df.columns = [str(c).strip() if c else c for c in df.columns]

# Normalizar columnas clave
df['Familia'] = df['Familia'].fillna('').astype(str).str.strip()
df['Descripción de Producto'] = df['Descripción de Producto'].fillna('').astype(str).str.strip()
df['Ramo'] = df.iloc[:, 20].fillna('').astype(str).str.strip()  # col 21 (0-indexed 20)

print(f'Productos cargados: {len(df)}')
print('Familias:', sorted(df['Familia'].unique()))

Productos cargados: 1437
Familias: ['ACCES. OTRAS M', 'AGR DEL. OTRAS', 'AGR TRAS. OTRAS', 'AGRICOLA DEL.', 'AGRICOLA TRAS.', 'BATERIAS', 'CAM MOTO OTRAS', 'CAMIONETA OTRAS', 'GIGANTE', 'GIGANTE OTRAS M', 'IND. Y VIALES', 'LUBRICANTES', 'MOTO', 'MOTO OTRAS M.', 'PASEO', 'PASEO OTRAS MAR', 'PICK UP', 'VIALES OTRAS M']


In [3]:
# ── Parsear tabla de precios de la hoja más reciente ──────────────────────
SHEET = 'Costo ag. Mayo 2026'  # ← cambiar para otro mes
ws_ag = wb_ag[SHEET]
ag_rows = list(ws_ag.iter_rows(values_only=True))

# Fila 6 (idx=5) = nombres de agencias; fila 7 (idx=6) = headers; fila 8+ = datos
agency_name_row = ag_rows[5]
header_row      = ag_rows[6]
data_rows       = ag_rows[7:]

# Detectar columnas por agencia (agrupadas: nombre en pos par, precio en pos impar)
ncols = len(agency_name_row)
agency_col_groups = {}   # agency_name -> [col_indices]
col = 0
while col < ncols:
    name = agency_name_row[col]
    if name and str(name).strip():
        next_agency = col + 1
        while next_agency < ncols and (agency_name_row[next_agency] is None or str(agency_name_row[next_agency]).strip() == ''):
            next_agency += 1
        agency_col_groups[str(name).strip()] = list(range(col, next_agency))
        col = next_agency
    else:
        col += 1

print('Agencias detectadas:')
for ag, cols_idx in agency_col_groups.items():
    hdrs = [header_row[c] for c in cols_idx if header_row[c]]
    print(f'  {ag}: cols {cols_idx} → {hdrs}')

Agencias detectadas:
  DAC: cols [0, 1] → ['Categorias', 'Precios iva incl']
  NASAZZI: cols [2, 3] → ['Categorias', 'Precios iva incl']
  MEGAM: cols [4, 5] → ['Categorias', 'Precios iva incl']
  Transportes Nagar sas (Expreso Ruta 1): cols [6, 7] → ['Categorias', 'Precios iva incl.  ']
  SELEGUIN: cols [8, 9] → ['Categorias', 'Precios iva incl. Actualizados']
  EXPRESO ROCHA: cols [10, 11] → ['Categorias', 'Precios iva incl.']
  BULEVAR (ACC): cols [12, 13] → ['Categorias', 'Precios iva incl.']
  PERICO: cols [14, 15] → ['Categorias', 'Precios iva incl.']
  TRUJILLO: cols [16, 17] → ['Categorias', 'Precios iva incl.']
  GONFER: cols [18, 19, 20, 21] → ['Categorias', 'Precios sin IVA (Empresa literal E)', '% aumento', 'Precio Actualizado sin IVA (Empresa literal E)']
  3EME (El Chambon): cols [22, 23] → ['Categorias', 'Precios iva incl.']
  Martin Escudero (Lascano): cols [24, 25] → ['Categorias ', 'total iva incl.']
  Arzuaga: cols [26, 27, 28] → ['Categorias ', ' sin levante  IVA in

In [4]:
# ── Extraer dict {agencia: {categoria: precio}} ──────────────────────────
# Para GONFER usamos col 3 del grupo (precio actualizado sin IVA)
# Para Arzuaga usamos col 1 del grupo (Young y Paysandú)

def pick_price_col(agency_name, cols_in_group, header_row):
    """Devuelve el índice de columna de categoría y precio para la agencia."""
    cat_col   = cols_in_group[0]
    price_col = cols_in_group[1] if len(cols_in_group) > 1 else cols_in_group[0]
    if 'GONFER' in agency_name.upper() and len(cols_in_group) >= 4:
        price_col = cols_in_group[3]  # Precio Actualizado sin IVA
    return cat_col, price_col

agency_prices = {}   # {agency: {categoria_str: precio_float}}
agency_meta   = {}   # {agency: 'IVA incl.' | 'sin IVA'}

for ag, cols_idx in agency_col_groups.items():
    cat_col, price_col = pick_price_col(ag, cols_idx, header_row)
    prices = {}
    for row in data_rows:
        cat   = row[cat_col]   if cat_col   < len(row) else None
        price = row[price_col] if price_col < len(row) else None
        if cat and isinstance(cat, str) and cat.strip() and price and isinstance(price, (int, float)):
            prices[cat.strip()] = float(price)
    agency_prices[ag] = prices
    agency_meta[ag] = 'sin IVA' if 'GONFER' in ag.upper() else 'IVA incl.'

# Mostrar resumen
for ag, prices in agency_prices.items():
    note = f'  [{agency_meta[ag]}]' if agency_meta[ag] == 'sin IVA' else ''
    print(f'{ag}{note}: {len(prices)} categorías')
    for k, v in list(prices.items())[:3]:
        print(f'    {k}: ${v:,.2f}')

DAC: 9 categorías
    Rodado 12,13,14: $296.77
    Rodado 15 a 18: $414.92
    Rodado 20 a 22,5: $445.17
NASAZZI: 20 categorías
    Rodado 13 y 14: $145.18
    Rodado 15 a 19: $196.42
    Rodado 20 a 22,5: $506.30
MEGAM: 9 categorías
    Bat y bultos: $100.00
    Cub 13 y 14: $60.00
    Cubierta auto (15 a 17): $90.00
Transportes Nagar sas (Expreso Ruta 1): 9 categorías
    Auto: $108.89
    Camioneta: $147.32
    Camion: $294.63
SELEGUIN: 12 categorías
    Baterias chicas: $140.30
    Baterias grandes: $219.60
    Cubierta auto: $114.68
EXPRESO ROCHA: 8 categorías
    Cubierta chica hasta 205/70.16: $122.00
    Cubierta grande hasta 750.16: $170.80
    Cubierta camion: $353.80
BULEVAR (ACC): 22 categorías
    Atado cub moto: $97.60
    Bat chicas: $146.40
    Bat grandes: $219.60
PERICO: 18 categorías
    Atado cub moto: $86.62
    Bat chicas: $129.32
    Bat grandes: $167.14
TRUJILLO: 11 categorías
    Bultos: $90.00
    Paseo hasta 16: $90.00
    Camioneta 17 en adelante: $100.00
GO

In [5]:
# ── CATEGORÍAS UNIFICADAS y MAPEO AGENCIA → CATEGORÍA ───────────────────
# Cada agencia usa nombres propios. Este dict traduce cada categoría unificada
# al label exacto de la tabla de precios de cada agencia.
# None = no cotiza ese tipo de producto.
# EDITAR si cambian los nombres.

UNIFIED_CATS = {
    'cub_moto':          'Cubierta Moto',
    'cub_auto_r12_r14':  'Cubierta Auto R12-R14',
    'cub_auto_r15_r19':  'Cubierta Auto/Camioneta R15-R19',
    'cub_camion_r20_r22':'Cubierta Camión R20-R22.5',
    'cub_agro_del':      'Cubierta Agrícola Delantera',
    'cub_agro_tras_med': 'Cubierta Agrícola Trasera Mediana 24-26"',
    'cub_agro_tras_gde': 'Cubierta Agrícola Trasera Grande 28-36"',
    'cub_agro_tras_xgde':'Cubierta Agrícola/Vial Extra Grande 38"+',
    'camara':            'Cámara (cualquier tamaño)',
    'bateria_chica':     'Batería Chica (≤110 Ah)',
    'bateria_grande':    'Batería Grande (>110 Ah)',
    'lubricante_caja':   'Lubricante en caja/lata',
    'lubricante_tambor': 'Lubricante Tambor 205L',
    'bulto_general':     'Bulto General',
}

MAPPING = {
    'DAC': {
        'cub_moto':          'Atado moto',
        'cub_auto_r12_r14':  'Rodado 12,13,14',
        'cub_auto_r15_r19':  'Rodado 15 a 18',
        'cub_camion_r20_r22':'Rodado 20 a 22,5',
        'cub_agro_del':      'Cubierta tractor AGRO 24 a 42',
        'cub_agro_tras_med': 'Cubierta tractor AGRO 24 a 42',
        'cub_agro_tras_gde': 'Cubierta tractor AGRO 24 a 42',
        'cub_agro_tras_xgde':'Cubierta tractor AGRO 24 a 42',
        'camara':            'Bulto hasta 15kg',
        'bateria_chica':     'Bateria chica',
        'bateria_grande':    'Bateria grande',
        'lubricante_caja':   'Bulto hasta 15kg',
        'lubricante_tambor': 'Tambor 205lts',
        'bulto_general':     'Bulto hasta 15kg',
    },
    'NASAZZI': {
        'cub_moto':          'Rodado 15 a 19',
        'cub_auto_r12_r14':  'Rodado 13 y 14',
        'cub_auto_r15_r19':  'Rodado 15 a 19',
        'cub_camion_r20_r22':'Rodado 20 a 22,5',
        'cub_agro_del':      'Rodado 24 y 25',
        'cub_agro_tras_med': 'Rodado 24 y 25',
        'cub_agro_tras_gde': 'Rodado 26 a 32',
        'cub_agro_tras_xgde':'Cub viales de 38 a 46 extra grande',
        'camara':            'Bultos',
        'bateria_chica':     'Bat hasta 110 amp',
        'bateria_grande':    'Bat mayor 110 amp',
        'lubricante_caja':   'Bultos lubricantes',
        'lubricante_tambor': 'Tambor 205lts',
        'bulto_general':     'Bultos',
    },
    'MEGAM': {
        'cub_moto':          'Bat y bultos',
        'cub_auto_r12_r14':  'Cub 13 y 14',
        'cub_auto_r15_r19':  'Cubierta auto (15 a 17)',
        'cub_camion_r20_r22':'Cubierta camion',
        'cub_agro_del':      'Tractor chico',
        'cub_agro_tras_med': 'Tractor chico',
        'cub_agro_tras_gde': 'Tractor grande',
        'cub_agro_tras_xgde':'Tractor grande',
        'camara':            'Bat y bultos',
        'bateria_chica':     'Bat y bultos',
        'bateria_grande':    'Bat y bultos',
        'lubricante_caja':   'Bat y bultos',
        'lubricante_tambor': 'Tambor 205lts',
        'bulto_general':     'Bat y bultos',
    },
    'Transportes Nagar sas (Expreso Ruta 1)': {
        'cub_moto':          'Auto',
        'cub_auto_r12_r14':  'Auto',
        'cub_auto_r15_r19':  'Camioneta',
        'cub_camion_r20_r22':'Camion',
        'cub_agro_del':      'Tractor del y 17,5',
        'cub_agro_tras_med': 'Tractor traseras',
        'cub_agro_tras_gde': 'Tractor grande',
        'cub_agro_tras_xgde':'Tractor grande',
        'camara':            'Bultos',
        'bateria_chica':     'Baterias',
        'bateria_grande':    'Baterias',
        'lubricante_caja':   'Bultos',
        'lubricante_tambor': 'Tambor',
        'bulto_general':     'Bultos',
    },
    'SELEGUIN': {
        'cub_moto':          'Bultos, atados, bolsas, cajas',
        'cub_auto_r12_r14':  'Cubierta auto',
        'cub_auto_r15_r19':  'Cubierta camioneta',
        'cub_camion_r20_r22':'Cubierta camion',
        'cub_agro_del':      'Bultos, atados, bolsas, cajas',
        'cub_agro_tras_med': 'Tractor mediana 24 a 26',
        'cub_agro_tras_gde': 'Tractor grande 28 a 30',
        'cub_agro_tras_xgde':'Tractor extra grande +30',
        'camara':            'Bultos, atados, bolsas, cajas',
        'bateria_chica':     'Baterias chicas',
        'bateria_grande':    'Baterias grandes',
        'lubricante_caja':   'Bultos, atados, bolsas, cajas',
        'lubricante_tambor': None,
        'bulto_general':     'Bultos, atados, bolsas, cajas',
    },
    'EXPRESO ROCHA': {
        'cub_moto':          'Bultos camaras',
        'cub_auto_r12_r14':  'Cubierta chica hasta 205/70.16',
        'cub_auto_r15_r19':  'Cubierta grande hasta 750.16',
        'cub_camion_r20_r22':'Cubierta camion',
        'cub_agro_del':      'Neumatico agricolas y viales hasta 18.4/15-26',
        'cub_agro_tras_med': 'Neum. Agricolas y viales mas de 23.1/18-26',
        'cub_agro_tras_gde': 'Neum. Agricolas y viales mas de 23.1/18-26',
        'cub_agro_tras_xgde':'Neum. Agricolas y viales mas de 23.1/18-26',
        'camara':            'Bultos camaras',
        'bateria_chica':     'Baterias',
        'bateria_grande':    'Baterias',
        'lubricante_caja':   'Bultos camaras',
        'lubricante_tambor': None,
        'bulto_general':     'Bultos camaras',
    },
    'BULEVAR (ACC)': {
        'cub_moto':          'Atado cub moto',
        'cub_auto_r12_r14':  'Cub auto  hasta rod 14',
        'cub_auto_r15_r19':  'Cub camioneta 15 a 19',
        'cub_camion_r20_r22':'Cub camion',
        'cub_agro_del':      'Cub tractor chica (hasta 22)',
        'cub_agro_tras_med': 'Cub tractor mediana (24)',
        'cub_agro_tras_gde': 'Cub tractor grande (de 25 a 36)',
        'cub_agro_tras_xgde':'Cub vial rod 38 a 42',
        'camara':            'Bolsas',
        'bateria_chica':     'Bat chicas',
        'bateria_grande':    'Bat grandes',
        'lubricante_caja':   'Caja aceite 1 y 4lts',
        'lubricante_tambor': 'Tambor 205lts',
        'bulto_general':     'Bolsas',
    },
    'PERICO': {
        'cub_moto':          'Atado cub moto',
        'cub_auto_r12_r14':  'Cub auto  hasta rod 14',
        'cub_auto_r15_r19':  'Cub camioneta 15 a 19',
        'cub_camion_r20_r22':'Cub camion',
        'cub_agro_del':      'Cub tractor chica',
        'cub_agro_tras_med': 'Cub tractor mediana',
        'cub_agro_tras_gde': 'Cub tractor grande',
        'cub_agro_tras_xgde':'Cub vial rod 38 a 42',
        'camara':            'Bolsas',
        'bateria_chica':     'Bat chicas',
        'bateria_grande':    'Bat grandes',
        'lubricante_caja':   'Caja aceite 1 y 4lts',
        'lubricante_tambor': 'Tambor 205lts',
        'bulto_general':     'Bolsas',
    },
    'TRUJILLO': {
        'cub_moto':          'Bultos',
        'cub_auto_r12_r14':  'Paseo hasta 16',
        'cub_auto_r15_r19':  'Paseo hasta 16',
        'cub_camion_r20_r22':'Camion',
        'cub_agro_del':      'Agricola delantera',
        'cub_agro_tras_med': 'Agricola mediana',
        'cub_agro_tras_gde': 'Agricola extra grande',
        'cub_agro_tras_xgde':'Agricola extra grande',
        'camara':            'Bultos',
        'bateria_chica':     'Bat hasta 150 amp',
        'bateria_grande':    'Bat + 150 amp',
        'lubricante_caja':   'Cajas lubricantes',
        'lubricante_tambor': 'Tambor 205lts',
        'bulto_general':     'Bultos',
    },
    'GONFER': {
        'cub_moto':          'Bultos',
        'cub_auto_r12_r14':  'Cubierta de auto',
        'cub_auto_r15_r19':  'Cubierta de camioneta',
        'cub_camion_r20_r22':'Cubierta de camion chico',
        'cub_agro_del':      'Agricola chica',
        'cub_agro_tras_med': 'Agricola hasta rod 28',
        'cub_agro_tras_gde': 'Agricola grande rod 30 en adelante',
        'cub_agro_tras_xgde':'Agricola grande rod 30 en adelante',
        'camara':            'Bultos',
        'bateria_chica':     'Bat hasta 110 amp',
        'bateria_grande':    'Bat mayor 110 amp',
        'lubricante_caja':   'Bultos lubricantes',
        'lubricante_tambor': 'Tambor 205lts',
        'bulto_general':     'Bultos',
    },
    '3EME (El Chambon)': {
        'cub_moto':          'Atado cub moto',
        'cub_auto_r12_r14':  'Cub auto  hasta rod 14',
        'cub_auto_r15_r19':  'Cub camioneta 15 a 19',
        'cub_camion_r20_r22':'Cub camion',
        'cub_agro_del':      'Cub tractor chica',
        'cub_agro_tras_med': 'Cub tractor mediana',
        'cub_agro_tras_gde': 'Cub tractor grande',
        'cub_agro_tras_xgde':'Cub vial rod 38 a 42',
        'camara':            'Bolsas',
        'bateria_chica':     'Bat chicas',
        'bateria_grande':    'Bat grandes',
        'lubricante_caja':   'Caja aceite 1 y 4lts',
        'lubricante_tambor': 'Tambor 205lts',
        'bulto_general':     'Bolsas',
    },
    'Martin Escudero (Lascano)': {
        'cub_moto':          'Auto',
        'cub_auto_r12_r14':  'Auto',
        'cub_auto_r15_r19':  'Camioneta',
        'cub_camion_r20_r22':'Camion',
        'cub_agro_del':      'Agricola chica',
        'cub_agro_tras_med': 'Agricola mediana',
        'cub_agro_tras_gde': 'Agricola grande',
        'cub_agro_tras_xgde':'Agricola grande',
        'camara':            'Bultos/atados',
        'bateria_chica':     'Baterias chicas',
        'bateria_grande':    'Baterias grandes',
        'lubricante_caja':   'Bultos lubricantes',
        'lubricante_tambor': 'Tambor aceite',
        'bulto_general':     'Bultos/atados',
    },
    'Arzuaga': {
        'cub_moto':          'Atado cubiertas moto',
        'cub_auto_r12_r14':  'cubiertas r13 r14',
        'cub_auto_r15_r19':  'cubiertas r16 r18',
        'cub_camion_r20_r22':'Cubiertas camion',
        'cub_agro_del':      'Cubiertas tractor chica',
        'cub_agro_tras_med': 'Cubiertas tractor mediana',
        'cub_agro_tras_gde': 'Cubiertas tractor grande',
        'cub_agro_tras_xgde':'Cubiertas tractor grande',
        'camara':            'Paquetes/cajas',
        'bateria_chica':     'Baterias chicas',
        'bateria_grande':    'Baterias camion',
        'lubricante_caja':   'Paquetes/cajas',
        'lubricante_tambor': 'Tanque aceite',
        'bulto_general':     'Paquetes/cajas',
    },
    'Franchi': {
        'cub_moto':          'Bultos',
        'cub_auto_r12_r14':  'Cub.auto',
        'cub_auto_r15_r19':  'Cub.camioneta',
        'cub_camion_r20_r22':'Cub.camion',
        'cub_agro_del':      'Bultos',
        'cub_agro_tras_med': 'Cub. agricola mediana',
        'cub_agro_tras_gde': 'Cub. agricola gde',
        'cub_agro_tras_xgde':'Cub. agricola gde',
        'camara':            'Bultos',
        'bateria_chica':     'Baterias chicas',
        'bateria_grande':    'Baterias grandes',
        'lubricante_caja':   'Bultos',
        'lubricante_tambor': None,
        'bulto_general':     'Bultos',
    },
}

print('Mapeo cargado para', len(MAPPING), 'agencias.')

Mapeo cargado para 14 agencias.


In [6]:
# ── Tabla comparativa por CATEGORÍA UNIFICADA ────────────────────────────
import unicodedata

IVA = 1.22  # GONFER cotiza sin IVA; convertimos para comparar en igualdad

def norm(s):
    """Normaliza para comparación robusta: sin acentos, minúsculas, sin espacios extra."""
    if s is None: return ''
    s = str(s).replace('\xa0', ' ')          # NBSP → espacio
    s = re.sub(r'\s+', ' ', s).strip().lower()
    # Eliminar marcas diacríticas (acentos)
    s = ''.join(c for c in unicodedata.normalize('NFD', s)
                if unicodedata.category(c) != 'Mn')
    return s

# Índice normalizado para búsqueda rápida
agency_prices_norm = {
    ag: {norm(k): v for k, v in prices.items()}
    for ag, prices in agency_prices.items()
}

def lookup_price(agency, unified_cat, with_iva_adjustment=True):
    cat_label = MAPPING.get(agency, {}).get(unified_cat)
    if cat_label is None:
        return None
    price = agency_prices_norm.get(agency, {}).get(norm(cat_label))
    if price and with_iva_adjustment and agency_meta.get(agency) == 'sin IVA':
        price = round(price * IVA, 2)
    return price

agencies_list = list(MAPPING.keys())

rows_comp = []
for ucat, ucat_name in UNIFIED_CATS.items():
    row = {'Categoría Unificada': ucat_name}
    prices_this_cat = {}
    for ag in agencies_list:
        p = lookup_price(ag, ucat)
        row[ag] = p
        if p is not None:
            prices_this_cat[ag] = p
    if prices_this_cat:
        best_ag  = min(prices_this_cat, key=prices_this_cat.get)
        row['MEJOR AGENCIA'] = best_ag
        row['PRECIO MÍNIMO'] = prices_this_cat[best_ag]
    else:
        row['MEJOR AGENCIA'] = None
        row['PRECIO MÍNIMO'] = None
    rows_comp.append(row)

df_comp = pd.DataFrame(rows_comp).set_index('Categoría Unificada')

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 250)
pd.set_option('display.float_format', '${:,.0f}'.format)
print('=== COMPARATIVA DE PRECIOS POR CATEGORÍA (Mayo 2026) ===')
print('Nota: GONFER convertido a IVA incl. (+22%). Arzuaga = sin cargo de levante.')
df_comp[['MEJOR AGENCIA','PRECIO MÍNIMO'] + agencies_list]

=== COMPARATIVA DE PRECIOS POR CATEGORÍA (Mayo 2026) ===
Nota: GONFER convertido a IVA incl. (+22%). Arzuaga = sin cargo de levante.


,MEJOR AGENCIA,PRECIO MÍNIMO,DAC,NASAZZI,MEGAM,Transportes Nagar sas (Expreso Ruta 1),SELEGUIN,EXPRESO ROCHA,BULEVAR (ACC),PERICO,TRUJILLO,GONFER,3EME (El Chambon),Martin Escudero (Lascano),Arzuaga,Franchi
Categoría Unificada,,,,,,,,,,,,,,,,
Cubierta Moto,PERICO,$87,$278,$196,$100,$109,$128,$152,$98,$87,$90,$139,$87,$122,$183,$100
Cubierta Auto R12-R14,MEGAM,$60,$297,$145,$60,$109,$115,$122,$98,$87,$90,$139,$87,$122,$88,$120
Cubierta Auto/Camioneta R15-R19,MEGAM,$90,$415,$196,$90,$147,$115,$171,$122,$101,$90,$211,$101,$146,$106,$150
Cubierta Camión R20-R22.5,MEGAM,$200,$445,$506,$200,$295,$268,$354,$244,$217,$200,$257,$217,$427,$232,$300
Cubierta Agrícola Delantera,Franchi,$100,"$1,485","$1,111",$200,$237,$128,$958,$234,$212,$180,$238,$212,$488,$464,$100
"Cubierta Agrícola Trasera Mediana 24-26""",MEGAM,$200,"$1,485","$1,111",$200,$653,$695,"$1,171",$549,$334,$445,$786,$334,"$1,098",$830,$700
"Cubierta Agrícola Trasera Grande 28-36""",MEGAM,$650,"$1,485","$1,836",$650,"$1,018","$1,122","$1,171","$1,098",$669,$665,"$1,449",$669,"$1,586","$1,000","$1,400"
"Cubierta Agrícola/Vial Extra Grande 38""+",MEGAM,$650,"$1,485","$3,989",$650,"$1,018","$1,330","$1,171","$1,342","$1,493",$665,"$1,449","$1,493","$1,586","$1,000","$1,400"
Cámara (cualquier tamaño),PERICO,$87,$369,$181,$100,$128,$128,$152,$98,$87,$90,$139,$87,$152,$146,$100


In [7]:
# ── Clasificar cada producto de Libro1 ───────────────────────────────────
def extract_rim(desc):
    desc = str(desc).upper()
    # R + número (ej. R15, R22.5)
    m = re.search(r'\bR\s*(\d{2,3}(?:\.\d)?)\b', desc)
    if m: return float(m.group(1))
    # -XX al final de medida (ej. 7.50-16, 14.9-24)
    m = re.search(r'-(\d{2,3})\b', desc)
    if m: return float(m.group(1))
    return None

def extract_amp(desc):
    m = re.search(r'\b(\d{2,3})A\b', str(desc).upper())
    return int(m.group(1)) if m else None

def classify(familia, desc):
    f = str(familia).upper()
    d = str(desc).upper()

    if 'LUBRICANTE' in f:
        return 'lubricante_tambor' if ('TAMBOR' in d or '205' in d) else 'lubricante_caja'

    if 'BATERIA' in f:
        amp = extract_amp(desc)
        if amp: return 'bateria_grande' if amp > 110 else 'bateria_chica'
        return 'bateria_chica'

    if 'ACCES' in f or 'CAM MOTO' in f:
        return 'camara'

    if 'MOTO' in f and 'CAM' not in f:
        return 'cub_moto'

    if 'GIGANTE' in f:
        return 'cub_camion_r20_r22'

    if 'VIAL' in f or 'IND.' in f:
        rim = extract_rim(desc)
        return 'cub_agro_tras_xgde' if (rim and rim >= 38) else 'cub_agro_tras_gde'

    if 'AGR DEL' in f or 'AGRICOLA DEL' in f:
        rim = extract_rim(desc)
        if rim:
            if rim >= 38: return 'cub_agro_tras_xgde'
            if rim >= 28: return 'cub_agro_tras_gde'
            if rim >= 24: return 'cub_agro_tras_med'
        return 'cub_agro_del'

    if 'AGR TRAS' in f or 'AGRICOLA TRAS' in f:
        rim = extract_rim(desc)
        if rim:
            if rim >= 38: return 'cub_agro_tras_xgde'
            if rim >= 30: return 'cub_agro_tras_gde'
            if rim >= 24: return 'cub_agro_tras_med'
        return 'cub_agro_tras_med'

    if any(x in f for x in ['PASEO', 'CAMIONETA', 'PICK UP']):
        rim = extract_rim(desc)
        if rim:
            if rim <= 14: return 'cub_auto_r12_r14'
            if rim <= 19: return 'cub_auto_r15_r19'
            return 'cub_camion_r20_r22'
        return 'cub_auto_r15_r19'

    return 'bulto_general'

df['categoria_unif'] = df.apply(
    lambda r: classify(r['Familia'], r['Descripción de Producto']), axis=1
)
df['categoria_nombre'] = df['categoria_unif'].map(UNIFIED_CATS)

print(df.groupby('categoria_nombre').size().sort_values(ascending=False).to_string())

categoria_nombre
Cubierta Auto/Camioneta R15-R19             538
Cubierta Camión R20-R22.5                   211
Cubierta Moto                               178
Cubierta Agrícola Trasera Mediana 24-26"    120
Cubierta Agrícola Trasera Grande 28-36"     115
Cámara (cualquier tamaño)                    86
Cubierta Agrícola/Vial Extra Grande 38"+     47
Cubierta Agrícola Delantera                  44
Cubierta Auto R12-R14                        34
Lubricante en caja/lata                      26
Batería Grande (>110 Ah)                     16
Batería Chica (≤110 Ah)                      13
Lubricante Tambor 205L                        9


In [8]:
# ── Precio de envío por producto × agencia ───────────────────────────────
for ag in agencies_list:
    df[ag] = df['categoria_unif'].apply(lambda c: lookup_price(ag, c))

# Mejor agencia y precio mínimo por producto
price_cols = agencies_list
df['PRECIO_MIN'] = df[price_cols].min(axis=1)
df['MEJOR_AGENCIA'] = df[price_cols].idxmin(axis=1)

# Columnas a mostrar
out_cols = ['Prod.', 'Descripción de Producto', 'Familia', 'categoria_nombre',
            'MEJOR_AGENCIA', 'PRECIO_MIN'] + agencies_list
df_out = df[out_cols].copy()

print(f'Productos clasificados: {len(df_out)}')
print('\nMejores agencias por cantidad de productos:')
print(df_out['MEJOR_AGENCIA'].value_counts())
df_out.head(20)

Productos clasificados: 1437

Mejores agencias por cantidad de productos:
MEJOR_AGENCIA
MEGAM       1116
PERICO       264
Franchi       44
TRUJILLO      13
Name: count, dtype: int64


,Prod.,Descripción de Producto,Familia,categoria_nombre,MEJOR_AGENCIA,PRECIO_MIN,DAC,NASAZZI,MEGAM,Transportes Nagar sas (Expreso Ruta 1),SELEGUIN,EXPRESO ROCHA,BULEVAR (ACC),PERICO,TRUJILLO,GONFER,3EME (El Chambon),Martin Escudero (Lascano),Arzuaga,Franchi
0,"$20,046",CAMARA SELETTO KR16 TR15,ACCES. OTRAS M,Cámara (cualquier tamaño),PERICO,$87,$369,$181,$100,$128,$128,$152,$98,$87,$90,$139,$87,$152,$146,$100
1,"$20,132",CAMARA SELETTO 14.9/13 -24 TR218A EX,ACCES. OTRAS M,Cámara (cualquier tamaño),PERICO,$87,$369,$181,$100,$128,$128,$152,$98,$87,$90,$139,$87,$152,$146,$100
2,"$20,052",CAMARA SELETTO 7.50 - 16 TR15 EX,ACCES. OTRAS M,Cámara (cualquier tamaño),PERICO,$87,$369,$181,$100,$128,$128,$152,$98,$87,$90,$139,$87,$152,$146,$100
3,"$20,085",CAMARA SELETTO 18.4 -34 TR218A EX,ACCES. OTRAS M,Cámara (cualquier tamaño),PERICO,$87,$369,$181,$100,$128,$128,$152,$98,$87,$90,$139,$87,$152,$146,$100
4,"$20,040",CAMARA SELETTO KR14 TR13,ACCES. OTRAS M,Cámara (cualquier tamaño),PERICO,$87,$369,$181,$100,$128,$128,$152,$98,$87,$90,$139,$87,$152,$146,$100
5,"$20,058",CAMARA SELETTO 12-16.5 TR15,ACCES. OTRAS M,Cámara (cualquier tamaño),PERICO,$87,$369,$181,$100,$128,$128,$152,$98,$87,$90,$139,$87,$152,$146,$100
6,"$20,048",CAMARA SELETTO 6.50/7.00-16 TR15,ACCES. OTRAS M,Cámara (cualquier tamaño),PERICO,$87,$369,$181,$100,$128,$128,$152,$98,$87,$90,$139,$87,$152,$146,$100
7,"$20,049",CAMARA SELETTO 7.50 R 16 TR75A,ACCES. OTRAS M,Cámara (cualquier tamaño),PERICO,$87,$369,$181,$100,$128,$128,$152,$98,$87,$90,$139,$87,$152,$146,$100
8,"$20,083",CAMARA SELETTO 11.2/12.4-24 TR218A EX,ACCES. OTRAS M,Cámara (cualquier tamaño),PERICO,$87,$369,$181,$100,$128,$128,$152,$98,$87,$90,$139,$87,$152,$146,$100
9,"$20,044",CAMARA SELETTO KR15 TR13,ACCES. OTRAS M,Cámara (cualquier tamaño),PERICO,$87,$369,$181,$100,$128,$128,$152,$98,$87,$90,$139,$87,$152,$146,$100


In [9]:
# ── Resumen por categoría: ranking de agencias ───────────────────────────
print('=== RANKING DE AGENCIAS POR CATEGORÍA ===')
for ucat, ucat_name in UNIFIED_CATS.items():
    prices = {}
    for ag in agencies_list:
        p = lookup_price(ag, ucat)
        if p is not None:
            prices[ag] = p
    if not prices:
        continue
    sorted_prices = sorted(prices.items(), key=lambda x: x[1])
    print(f'\n{ucat_name}:')
    for i, (ag, p) in enumerate(sorted_prices, 1):
        note = ' ← MEJOR' if i == 1 else ''
        print(f'  {i:2}. {ag:<45} ${p:>10,.2f}{note}')

=== RANKING DE AGENCIAS POR CATEGORÍA ===

Cubierta Moto:
   1. PERICO                                        $     86.62 ← MEJOR
   2. 3EME (El Chambon)                             $     86.62
   3. TRUJILLO                                      $     90.00
   4. BULEVAR (ACC)                                 $     97.60
   5. MEGAM                                         $    100.00
   6. Franchi                                       $    100.00
   7. Transportes Nagar sas (Expreso Ruta 1)        $    108.89
   8. Martin Escudero (Lascano)                     $    122.00
   9. SELEGUIN                                      $    128.10
  10. GONFER                                        $    139.08
  11. EXPRESO ROCHA                                 $    152.50
  12. Arzuaga                                       $    183.00
  13. NASAZZI                                       $    196.42
  14. DAC                                           $    278.05

Cubierta Auto R12-R14:
   1. MEGAM   

In [10]:
# ── Exportar a Excel ─────────────────────────────────────────────────────
output_file = 'comparativa_resultado.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Hoja 1: Comparativa por categoría
    df_comp.to_excel(writer, sheet_name='Por Categoría')

    # Hoja 2: Todos los productos con precios
    df_out.to_excel(writer, sheet_name='Por Producto', index=False)

    # Hoja 3: Resumen por familia
    df_fam = df_out.groupby(['Familia', 'categoria_nombre', 'MEJOR_AGENCIA'])['PRECIO_MIN'].mean().reset_index()
    df_fam.columns = ['Familia', 'Categoría Unificada', 'Mejor Agencia', 'Precio Prom. Mínimo']
    df_fam.to_excel(writer, sheet_name='Por Familia', index=False)

print(f'Exportado: {output_file}')
print('  - Hoja "Por Categoría": tabla comparativa de precios por categoría unificada')
print('  - Hoja "Por Producto":  cada producto con el costo de envío por agencia')
print('  - Hoja "Por Familia":   resumen agrupado por familia de productos')

Exportado: comparativa_resultado.xlsx
  - Hoja "Por Categoría": tabla comparativa de precios por categoría unificada
  - Hoja "Por Producto":  cada producto con el costo de envío por agencia
  - Hoja "Por Familia":   resumen agrupado por familia de productos
